Import your API keys through dotenv using load_dotenv

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

Import all the required Libraries 

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

Load your document or pdf you want to use

In [5]:
loader = PyPDFLoader("gate_syllabus.pdf")
docs = loader.load()
len(docs)



1

Split your doc/texts using langchain-text-splitters

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)


4

In [7]:
embeddingss = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [8]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddingss
)

In [9]:
query = "Linear Algebra and Machine Learning"
data = vector_store.similarity_search(query=query)

In [10]:
context = ""
for doc in data:
    context += doc.page_content + "\n"

In [11]:
LLM = ChatGroq(model="openai/gpt-oss-20b")



In [12]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {

        "context":context,
        "question":query
    }

In [13]:
prompt = PromptTemplate.from_template("""
You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [14]:
rag_chain = get_context | prompt | LLM

In [ ]:
res = rag_chain.invoke("What is the syllabus of Machine Learning ? ")


In [17]:
print(res.content)

**Machine Learning Syllabus (as outlined in the provided context)**  

| **Area** | **Key Topics** |
|----------|----------------|
| **1. Foundations** | • Probability & Statistics (confidence intervals, hypothesis tests, Bayes theorem) <br>• Linear Algebra (vectors, matrices, projections, eigen‑values, SVD) <br>• Calculus & Optimization (single‑variable functions, maxima/minima, gradient descent) |
| **2. Supervised Learning** | • **Regression**: simple & multiple linear regression, ridge regression, logistic regression (binary & multiclass) <br>• **Classification**: k‑Nearest Neighbour, Naïve Bayes, Linear Discriminant Analysis <br>• **Model Assessment**: bias‑variance trade‑off, cross‑validation (LOO, k‑fold), confidence intervals for predictions |
| **3. Unsupervised Learning** | • Clustering: k‑means / k‑medoid, hierarchical clustering (top‑down / bottom‑up), single‑linkage, multiple‑linkage <br>• Dimensionality Reduction: Principal Component Analysis (PCA) |
| **4. Advanced Model